In [1]:
!pip install fastdtw pandas numpy scipy scikit-learn xgboost lightgbm catboost


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import time
import warnings
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score, recall_score, precision_score

warnings.filterwarnings('ignore')

**1. TIỀN XỬ LÝ (LỌC NHIỄU, TÁI TẠO, VÀ CHUẨN HÓA 2D)**

In [3]:
# [NOISE ISOLATION] - Lọc video đứng yên
def is_dancing(df_student, variance_threshold=0.015):
    key_joints = ['left_wrist', 'right_wrist', 'left_ankle', 'right_ankle']
    df_key = df_student[df_student['Joint_Name'].isin(key_joints)]
    movement_std = df_key.groupby('Joint_Name')[['X', 'Y']].std().mean().mean()
    return movement_std >= variance_threshold, movement_std

# [SIGNAL RECONSTRUCTION & ANATOMICAL SCALING] - Khử nhiễu trục Z và dời tâm
def preprocess_data(df):
    df = df.sort_values(by=['Frame', 'Joint_Name']).reset_index(drop=True)
    mask = df['Visibility'] < 0.5
    df.loc[mask, ['X', 'Y', 'Z']] = np.nan
    df[['X', 'Y', 'Z']] = df.groupby('Joint_Name')[['X', 'Y', 'Z']].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    df.fillna(0, inplace=True)
    
    normalized_frames = []
    for frame_id, frame_data in df.groupby('Frame'):
        l_hip = frame_data[frame_data['Joint_Name'] == 'left_hip']
        r_hip = frame_data[frame_data['Joint_Name'] == 'right_hip']
        l_sh = frame_data[frame_data['Joint_Name'] == 'left_shoulder']
        r_sh = frame_data[frame_data['Joint_Name'] == 'right_shoulder']
        
        if not l_hip.empty and not r_hip.empty and not l_sh.empty and not r_sh.empty:
            root_x = (l_hip['X'].values[0] + r_hip['X'].values[0]) / 2
            root_y = (l_hip['Y'].values[0] + r_hip['Y'].values[0]) / 2
            sh_x = (l_sh['X'].values[0] + r_sh['X'].values[0]) / 2
            sh_y = (l_sh['Y'].values[0] + r_sh['Y'].values[0]) / 2
            
            torso_length = np.sqrt((sh_x - root_x)**2 + (sh_y - root_y)**2)
            torso_length = max(torso_length, 1e-5)
        else:
            root_x, root_y = 0, 0
            torso_length = 1.0
            
        frame_data = frame_data.copy()
        frame_data['X_norm'] = (frame_data['X'] - root_x) / torso_length
        frame_data['Y_norm'] = (frame_data['Y'] - root_y) / torso_length
        frame_data['Z_norm'] = frame_data['Z']
        
        normalized_frames.append(frame_data)
        
    return pd.concat(normalized_frames, ignore_index=True)

**2. XỬ LÝ ĐẶC TRƯNG (KHOẢNG CÁCH, GÓC ĐỘ VÀ VẬN TỐC)**

In [4]:
# [FEATURE EXTRACTION] - Trích xuất 3 Đặc trưng
def extract_advanced_features(df_student, df_teacher):
    def get_vectors(df):
        wide = df.pivot(index='Frame', columns='Joint_Name', values=['X_norm', 'Y_norm']).fillna(0)
        return wide.values, wide.index.tolist()

    vec_s, frames_s = get_vectors(df_student)
    vec_t, frames_t = get_vectors(df_teacher)
    
    distance, path = fastdtw(vec_s, vec_t, dist=euclidean)
    aligned_pairs = [{'Frame_s': frames_s[s], 'Frame_t': frames_t[t]} for s, t in path]
    df_path = pd.DataFrame(aligned_pairs)
    
    df_s_aligned = pd.merge(df_path, df_student, left_on='Frame_s', right_on='Frame').drop(columns=['Frame'])
    df_t_aligned = pd.merge(df_path, df_teacher, left_on='Frame_t', right_on='Frame').drop(columns=['Frame'])
    
    df_s_aligned = df_s_aligned.rename(columns={'X_norm': 'X_s', 'Y_norm': 'Y_s'})
    df_t_aligned = df_t_aligned.rename(columns={'X_norm': 'X_t', 'Y_norm': 'Y_t'})
    
    label_col = 'Label' if 'Label' in df_s_aligned.columns else ('Label_s' if 'Label_s' in df_s_aligned.columns else None)
    cols_to_merge_s = ['Frame_s', 'Frame_t', 'Joint_Name', 'X_s', 'Y_s'] + ([label_col] if label_col else [])
    
    df_merged = pd.merge(df_s_aligned[cols_to_merge_s], 
                         df_t_aligned[['Frame_s', 'Frame_t', 'Joint_Name', 'X_t', 'Y_t']], 
                         on=['Frame_s', 'Frame_t', 'Joint_Name'])
    
    # Đặc trưng 1 & 2: Error & Cosine
    df_merged['Joint_Error'] = np.sqrt((df_merged['X_s'] - df_merged['X_t'])**2 + (df_merged['Y_s'] - df_merged['Y_t'])**2)
    dot = (df_merged['X_s']*df_merged['X_t']) + (df_merged['Y_s']*df_merged['Y_t'])
    norm_s = np.sqrt(df_merged['X_s']**2 + df_merged['Y_s']**2)
    norm_t = np.sqrt(df_merged['X_t']**2 + df_merged['Y_t']**2)
    df_merged['Joint_Cosine'] = dot / (norm_s * norm_t + 1e-9)
    
    # Đặc trưng 3: Velocity Error (Vận tốc)
    df_merged['Vel_X_s'] = df_merged.groupby('Joint_Name')['X_s'].diff().fillna(0)
    df_merged['Vel_Y_s'] = df_merged.groupby('Joint_Name')['Y_s'].diff().fillna(0)
    df_merged['Vel_X_t'] = df_merged.groupby('Joint_Name')['X_t'].diff().fillna(0)
    df_merged['Vel_Y_t'] = df_merged.groupby('Joint_Name')['Y_t'].diff().fillna(0)
    
    df_merged['Velocity_Error'] = np.sqrt((df_merged['Vel_X_s'] - df_merged['Vel_X_t'])**2 + 
                                          (df_merged['Vel_Y_s'] - df_merged['Vel_Y_t'])**2)
    
    if label_col:
        df_merged.rename(columns={label_col: 'Ground_Truth_Label'}, inplace=True)
        
    return df_merged[['Frame_s', 'Joint_Name', 'Joint_Error', 'Joint_Cosine', 'Velocity_Error', 'Ground_Truth_Label']]

**3. TRÍCH XUẤT DATASET TỪ CÁC VIDEO**

In [5]:
dance_pairs = [
    {"id": 1, "student": "student_dance_1.csv", "teacher": "teacher_dance_1.csv"},
    {"id": 2, "student": "student_dance_2.csv", "teacher": "teacher_dance_2.csv"},
    {"id": 3, "student": "student_dance_3.csv", "teacher": "teacher_dance_3.csv"},
    {"id": 4, "student": "student_dance_4.csv", "teacher": "teacher_dance_4.csv"},
    {"id": 5, "student": "student_dance_5.csv", "teacher": "teacher_dance_5.csv"},
    {"id": 6, "student": "student_dance_6.csv", "teacher": "teacher_dance_6.csv"},
    {"id": 7, "student": "student_dance_7.csv", "teacher": "teacher_dance_7.csv"},
    {"id": 8, "student": "student_dance_8.csv", "teacher": "teacher_dance_8.csv"},
    {"id": 9, "student": "student_dance_9.csv", "teacher": "teacher_dance_9.csv"},
    {"id": 10, "student": "student_dance_10.csv", "teacher": "teacher_dance_10.csv"},
    {"id": 11, "student": "student_dance_11.csv", "teacher": "teacher_dance_11.csv"},
    {"id": 12, "student": "student_dance_12.csv", "teacher": "teacher_dance_12.csv"},
    {"id": 13, "student": "teacher_dance_12.csv", "teacher": "teacher_dance_12.csv"},
    {"id": 15, "student": "student_dance_15.csv", "teacher": "teacher_dance_15.csv"}
]

TEST_VIDEO_IDS = [5, 9] 
all_joints_data = []

print("=== BẮT ĐẦU TRÍCH XUẤT ĐẶC TRƯNG TỪ VIDEO ===")
for pair in dance_pairs:
    try:
        df_s = pd.read_csv(pair["student"])
        df_t = pd.read_csv(pair["teacher"])
        
        is_moving, std = is_dancing(df_s)
        if not is_moving:
            print(f"[BỘ LỌC] Bỏ qua Video {pair['id']}: Đứng yên (Std={std:.4f})")
            continue
            
        print(f"[V] Đang xử lý Video {pair['id']}...")
        df_s_clean = preprocess_data(df_s)
        df_t_clean = preprocess_data(df_t)
        
        joint_features_df = extract_advanced_features(df_s_clean, df_t_clean)
        joint_features_df['Video_ID'] = pair['id']
        all_joints_data.append(joint_features_df)
        
    except Exception as e:
        print(f"[-] Lỗi tại Video {pair['id']}: {e}")

master_dataset = pd.concat(all_joints_data, ignore_index=True)

# TÁCH TRAIN / TEST
train_data = master_dataset[~master_dataset['Video_ID'].isin(TEST_VIDEO_IDS)]
test_data = master_dataset[master_dataset['Video_ID'].isin(TEST_VIDEO_IDS)].copy()

print("\n======================================================")
print("             THỐNG KÊ TẬP DỮ LIỆU                     ")
print("======================================================")
print(f"- Số lượng Frame tập TRAIN  : {train_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]:,} frames")
print(f"- Số lượng Frame tập TEST   : {test_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]:,} frames")
print("======================================================\n")

=== BẮT ĐẦU TRÍCH XUẤT ĐẶC TRƯNG TỪ VIDEO ===
[V] Đang xử lý Video 1...
[V] Đang xử lý Video 2...
[V] Đang xử lý Video 3...
[V] Đang xử lý Video 4...
[V] Đang xử lý Video 5...
[V] Đang xử lý Video 6...
[V] Đang xử lý Video 7...
[V] Đang xử lý Video 8...
[V] Đang xử lý Video 9...
[V] Đang xử lý Video 10...
[V] Đang xử lý Video 11...
[V] Đang xử lý Video 12...
[V] Đang xử lý Video 13...
[V] Đang xử lý Video 15...

             THỐNG KÊ TẬP DỮ LIỆU                     
- Số lượng Frame tập TRAIN  : 6,533 frames
- Số lượng Frame tập TEST   : 613 frames



**4. SO SÁNH ĐA MÔ HÌNH VÀ ENSEMBLE LEARNING (CẤP ĐỘ KHỚP)**

In [6]:
features = ['Joint_Error', 'Joint_Cosine', 'Velocity_Error'] 
X_train_comp = train_data[features]
y_train_comp = train_data['Ground_Truth_Label']
X_test_comp = test_data[features]
y_test_comp = test_data['Ground_Truth_Label']

# Khởi tạo các mô hình cơ sở
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, random_state=42, verbose=0)
}

# Voting Ensemble (Bỏ KNN do khác bản chất với Tree-based)
voting_clf = VotingClassifier(
    estimators=[(name, model) for name, model in models.items() if name != 'KNN'],
    voting='soft' 
)
models['Ensemble (Voting)'] = voting_clf

print("=== BẢNG SO SÁNH HIỆU NĂNG CÁC THUẬT TOÁN HỌC MÁY (CHẤM CẤP ĐỘ KHỚP) ===\n")
print(f"{'Tên Mô hình':<20} | {'Độ chính xác (Acc)':<20} | {'F1-Score (Macro)':<18} | {'Thời gian Train'}")
print("-" * 80)

best_model_name = ""
best_f1 = 0
trained_models = {}

for name, model in models.items():
    start_time = time.time()
    model.fit(X_train_comp, y_train_comp)
    trained_models[name] = model
    
    y_pred_joint = model.predict(X_test_comp)
    
    acc = accuracy_score(y_test_comp, y_pred_joint)
    f1 = f1_score(y_test_comp, y_pred_joint, average='macro')
    train_time = time.time() - start_time
    
    print(f"{name:<20} | {acc*100:<18.2f}% | {f1:<18.4f} | {train_time:.3f} giây")
    
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name

print("-" * 80)
print(f"-> Mô hình chiến thắng: {best_model_name.upper()} ({best_f1:.4f})")

# Gán kết quả của mô hình tốt nhất vào dữ liệu Test
test_data['Predicted_Joint_Label'] = trained_models[best_model_name].predict(X_test_comp)

=== BẢNG SO SÁNH HIỆU NĂNG CÁC THUẬT TOÁN HỌC MÁY (CHẤM CẤP ĐỘ KHỚP) ===

Tên Mô hình          | Độ chính xác (Acc)   | F1-Score (Macro)   | Thời gian Train
--------------------------------------------------------------------------------
Decision Tree        | 71.87             % | 0.7042             | 0.259 giây
KNN                  | 61.28             % | 0.6037             | 0.078 giây
Random Forest        | 73.84             % | 0.7206             | 7.021 giây
Gradient Boosting    | 74.46             % | 0.7249             | 21.516 giây
XGBoost              | 72.76             % | 0.7108             | 1.672 giây
LightGBM             | 73.48             % | 0.7174             | 2.029 giây
CatBoost             | 74.15             % | 0.7235             | 0.782 giây
Ensemble (Voting)    | 73.59             % | 0.7181             | 34.690 giây
--------------------------------------------------------------------------------
-> Mô hình chiến thắng: GRADIENT BOOSTING (0.7249)


**5. RULE-BASED HEURISTICS & BÁO CÁO CẤP ĐỘ KHUNG HÌNH**

In [7]:
import time
import pandas as pd
import numpy as np

print("==================================================================")
print("     ĐÁNH GIÁ TỐC ĐỘ SUY LUẬN (INFERENCE TIME) TRÊN 1 FRAME       ")
print("==================================================================\n")

# 1. Trích xuất đúng 1 khung hình (1 Frame = 13 điểm khớp) từ tập Test
# Lấy khung hình đầu tiên của học sinh để làm mẫu test tốc độ
sample_frame_df = list(test_data.groupby(['Video_ID', 'Frame_s']))[0][1]

# Lấy các đặc trưng đầu vào (2 hoặc 3 đặc trưng tùy cậu đang dùng bản nào)
feature_cols = [col for col in ['Joint_Error', 'Joint_Cosine', 'Velocity_Error'] if col in test_data.columns]
X_single_frame = sample_frame_df[feature_cols]

# Cấu hình số vòng lặp để lấy trung bình (Tránh nhiễu do CPU máy tính chạy ngầm)
n_iterations = 1000 

print(f"{'Tên Mô hình':<20} | {'Thời gian/Frame (ms)':<22} | {'Tốc độ (FPS)':<15}")
print("-" * 65)

# 2. Đo tốc độ cho từng mô hình đã huấn luyện
for name, model in trained_models.items():
    
    # Warm-up (Khởi động mô hình vài lần để đưa model vào cache của RAM/CPU)
    for _ in range(10):
        _ = model.predict(X_single_frame)

    # Đo thời gian chính thức bằng công tơ mét hiệu năng cao (perf_counter)
    start_time = time.perf_counter()
    
    for _ in range(n_iterations):
        _ = model.predict(X_single_frame)
        
    end_time = time.perf_counter()

    # 3. Tính toán kết quả
    total_time = end_time - start_time
    time_per_frame_ms = (total_time / n_iterations) * 1000 # Đổi ra mili-giây (ms)
    
    # Tính FPS (Số khung hình xử lý được trong 1 giây)
    fps = 1000 / time_per_frame_ms if time_per_frame_ms > 0 else 0

    print(f"{name:<20} | {time_per_frame_ms:<15.4f} ms | {fps:<10.0f} FPS")

print("-" * 65)
print("* Yêu cầu Real-time tiêu chuẩn: Phải đạt > 30 FPS (Tức là < 33.3 ms / Frame).")

     ĐÁNH GIÁ TỐC ĐỘ SUY LUẬN (INFERENCE TIME) TRÊN 1 FRAME       

Tên Mô hình          | Thời gian/Frame (ms)   | Tốc độ (FPS)   
-----------------------------------------------------------------
Decision Tree        | 0.5856          ms | 1708       FPS
KNN                  | 1.2400          ms | 806        FPS
Random Forest        | 2.5587          ms | 391        FPS
Gradient Boosting    | 0.4652          ms | 2150       FPS
XGBoost              | 1.0097          ms | 990        FPS
LightGBM             | 1.3333          ms | 750        FPS
CatBoost             | 0.4892          ms | 2044       FPS
Ensemble (Voting)    | 18.5759         ms | 54         FPS
-----------------------------------------------------------------
* Yêu cầu Real-time tiêu chuẩn: Phải đạt > 30 FPS (Tức là < 33.3 ms / Frame).
